In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
digits = load_digits()

X = pd.DataFrame(digits.data)
y = digits.target

print(X.shape)

(1797, 64)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
num_cols = X.columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols)
])

In [5]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("pca", PCA()),
    ("svc", SVC())
])

In [6]:
param_grid = {
    "pca__n_components": [15, 20, 25, 30],
    "svc__C": [0.1, 1, 5],
    "svc__gamma": [0.0005, 0.001, 0.005],
    "svc__kernel": ["rbf"]
}

In [7]:
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [8]:
grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

Best Params: {'pca__n_components': 30, 'svc__C': 5, 'svc__gamma': 0.005, 'svc__kernel': 'rbf'}


In [9]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

In [10]:
print("Train Accuracy:", best_model.score(X_train, y_train))
print("Test Accuracy:", best_model.score(X_test, y_test))

Train Accuracy: 0.9958246346555324
Test Accuracy: 0.9833333333333333


In [11]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[33  0  0  0  0  0  0  0  0  0]
 [ 0 28  0  0  0  0  0  0  0  0]
 [ 0  0 33  0  0  0  0  0  0  0]
 [ 0  0  0 33  0  1  0  0  0  0]
 [ 0  0  0  0 46  0  0  0  0  0]
 [ 0  0  0  0  0 45  1  0  0  1]
 [ 0  0  0  0  0  1 34  0  0  0]
 [ 0  0  0  0  0  0  0 33  0  1]
 [ 0  0  0  0  0  0  0  0 30  0]
 [ 0  0  0  0  0  0  0  1  0 39]]


In [12]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        28
           2       1.00      1.00      1.00        33
           3       1.00      0.97      0.99        34
           4       1.00      1.00      1.00        46
           5       0.96      0.96      0.96        47
           6       0.97      0.97      0.97        35
           7       0.97      0.97      0.97        34
           8       1.00      1.00      1.00        30
           9       0.95      0.97      0.96        40

    accuracy                           0.98       360
   macro avg       0.99      0.98      0.98       360
weighted avg       0.98      0.98      0.98       360



In [13]:
pca = best_model.named_steps["pca"]
print(np.cumsum(pca.explained_variance_ratio_))

[0.12038006 0.21771104 0.30332925 0.36826702 0.41714616 0.45978663
 0.49928022 0.53277655 0.56301219 0.59233154 0.62005188 0.64549785
 0.66852848 0.6913693  0.71258671 0.73154928 0.74902345 0.76506686
 0.78091696 0.7960052  0.80940266 0.82213528 0.8335177  0.84395064
 0.85380441 0.86317825 0.87170997 0.88007258 0.88804326 0.89561788]
